# CRM Access Governance & Customer Data Protection
## Stage 6 — Data Classification & Privacy Engineering

This notebook extends the CRM Governance project into **customer data protection and privacy engineering**.

The current access-governance dataset does not contain real customer-level PII. Therefore, this stage creates a **synthetic customer layer** to simulate how a governed CRM analytics environment should handle personal data.

### Main Goals

1. Create a synthetic CRM customer dataset.
2. Classify fields as personal/non-personal data.
3. Distinguish direct identifiers, indirect identifiers, behavioral data, and governance attributes.
4. Define sensitivity levels.
5. Apply data minimization.
6. Apply masking and pseudonymization.
7. Create an analytics-safe customer layer.
8. Document privacy controls and their rationale.
9. Link privacy controls to the access-governance framework.
10. Prepare the project for future lineage, privacy monitoring, and AI Governance.

> **Important:** this is a simulated technical governance exercise. It does not constitute legal advice and does not claim that every field classification maps directly to an official LGPD/GDPR legal category.


In [ ]:
# 1. Libraries and Settings

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment ready.")


# 2. Why a Synthetic Customer Layer Is Needed

The original dataset focuses on **who accesses the CRM and under which conditions**.

To demonstrate privacy engineering, we also need to simulate **what customer data the CRM contains**.

This stage introduces a synthetic customer table with attributes such as:

- customer identifier;
- name;
- email;
- phone;
- date of birth;
- region;
- signup date;
- marketing consent;
- customer segment;
- purchase metrics.

The objective is not to create realistic personal identities. The goal is to build a controlled environment for privacy-preserving transformations.


# 3. Create Synthetic Customer Data

In [ ]:
n_customers = 10000

customer_ids = [f"CUST_{i:06d}" for i in range(1, n_customers + 1)]

first_names = [
    "Alex", "Jordan", "Taylor", "Morgan", "Casey",
    "Riley", "Jamie", "Avery", "Cameron", "Drew"
]

last_names = [
    "Silva", "Santos", "Oliveira", "Pereira", "Costa",
    "Rodrigues", "Almeida", "Nascimento", "Lima", "Souza"
]

states = ["SP", "RJ", "MG", "DF", "PR", "RS", "BA", "SC", "GO", "PE"]

segments = [
    "New Customer",
    "Regular",
    "High Value",
    "At Risk",
    "Inactive"
]

signup_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2022-01-01", "2026-08-01", freq="D"),
        size=n_customers
    )
)

birth_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("1955-01-01", "2005-12-31", freq="D"),
        size=n_customers
    )
)

customers = pd.DataFrame({
    "Customer_ID": customer_ids,
    "First_Name": np.random.choice(first_names, n_customers),
    "Last_Name": np.random.choice(last_names, n_customers),
    "Email": [
        f"customer{i}@example.com"
        for i in range(1, n_customers + 1)
    ],
    "Phone": [
        f"+55 11 9{np.random.randint(1000, 9999)}-{np.random.randint(1000, 9999)}"
        for _ in range(n_customers)
    ],
    "Birth_Date": birth_dates,
    "State": np.random.choice(states, n_customers),
    "Signup_Date": signup_dates,
    "Marketing_Consent": np.random.choice(
        [True, False],
        n_customers,
        p=[0.72, 0.28]
    ),
    "Customer_Segment": np.random.choice(
        segments,
        n_customers,
        p=[0.15, 0.45, 0.15, 0.15, 0.10]
    ),
    "Purchase_Count": np.random.poisson(8, n_customers),
    "Total_Revenue": np.round(
        np.random.gamma(2.5, 450, n_customers),
        2
    )
})

customers["Average_Ticket"] = np.where(
    customers["Purchase_Count"] > 0,
    customers["Total_Revenue"] / customers["Purchase_Count"],
    0
).round(2)

display(customers.head())
print(f"Rows: {len(customers):,}")


# 4. Field-Level Privacy Classification

Fields are classified by their privacy role.

### Proposed Classes

- `Direct Identifier`
- `Indirect Identifier`
- `Behavioral / Transactional`
- `Governance / Consent`
- `Non-Personal Analytical Attribute`

This classification is designed for the project and is not intended as a legal taxonomy.


In [ ]:
privacy_classification = pd.DataFrame([
    ["Customer_ID", "Direct Identifier", "Restricted", True, "Pseudonymize"],
    ["First_Name", "Direct Identifier", "Restricted", False, "Remove"],
    ["Last_Name", "Direct Identifier", "Restricted", False, "Remove"],
    ["Email", "Direct Identifier", "Restricted", False, "Mask or Remove"],
    ["Phone", "Direct Identifier", "Restricted", False, "Mask or Remove"],
    ["Birth_Date", "Indirect Identifier", "Confidential", False, "Generalize"],
    ["State", "Indirect Identifier", "Internal", True, "Keep"],
    ["Signup_Date", "Behavioral / Transactional", "Internal", True, "Keep"],
    ["Marketing_Consent", "Governance / Consent", "Restricted", True, "Keep"],
    ["Customer_Segment", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Purchase_Count", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Total_Revenue", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Average_Ticket", "Behavioral / Transactional", "Confidential", True, "Keep"]
], columns=[
    "Field_Name",
    "Privacy_Class",
    "Sensitivity",
    "Required_for_Analytics",
    "Recommended_Treatment"
])

display(privacy_classification)


# 5. Direct vs. Indirect Identifiers

### Direct Identifiers
Can directly point to an individual.

Examples:
- name;
- email;
- phone;
- customer identifier.

### Indirect Identifiers
May contribute to re-identification when combined with other attributes.

Examples:
- date of birth;
- location;
- rare demographic combinations.

This distinction helps determine whether a field should be removed, generalized, masked, or retained.


# 6. Data Minimization Matrix

The analytical layer should include only the data necessary for its intended purpose.

The proposed analytical use case is:

> Customer segmentation, CRM governance monitoring, consent analysis, and aggregate commercial analytics.

Fields that are not required for these purposes should not be exposed to the analytical layer.


In [ ]:
minimization_matrix = privacy_classification[
    [
        "Field_Name",
        "Privacy_Class",
        "Sensitivity",
        "Required_for_Analytics",
        "Recommended_Treatment"
    ]
].copy()

display(minimization_matrix)


# 7. Pseudonymization Function

The function below converts the original customer identifier into a deterministic pseudonymous token.

A deterministic hash allows the same customer to be linked across governed datasets without exposing the original identifier.

> In a real environment, secure key management, salts, HMAC, encryption, or tokenization services would normally be considered instead of plain hashing alone.


In [ ]:
def pseudonymize_identifier(value, salt="crm-governance-demo"):
    raw_value = f"{salt}|{value}"

    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()[:16]

customers["Customer_Token"] = (
    customers["Customer_ID"]
    .apply(pseudonymize_identifier)
)

display(
    customers[
        ["Customer_ID", "Customer_Token"]
    ].head()
)


## Study Note — Hashing vs. Encryption

### Hashing
- one-way transformation;
- generally not intended to be reversed;
- useful for pseudonymous linking when designed correctly.

### Encryption
- reversible with the appropriate key;
- useful when authorized systems must recover the original value.

### Tokenization
- replaces the original value with a surrogate token;
- mapping is usually maintained in a protected token vault.

For this portfolio project, hashing is used only to demonstrate the concept of pseudonymization.


# 8. Email Masking

Masked values preserve limited operational context while reducing direct exposure.


In [ ]:
def mask_email(email):
    if pd.isna(email) or "@" not in email:
        return None

    local, domain = email.split("@", 1)

    if len(local) <= 1:
        masked_local = "*"
    else:
        masked_local = local[0] + "***"

    return f"{masked_local}@{domain}"

customers["Email_Masked"] = (
    customers["Email"]
    .apply(mask_email)
)

display(
    customers[
        ["Email", "Email_Masked"]
    ].head()
)


# 9. Phone Masking

In [ ]:
def mask_phone(phone):
    if pd.isna(phone):
        return None

    digits = "".join(
        char for char in str(phone)
        if char.isdigit()
    )

    if len(digits) < 4:
        return "****"

    return "*" * (len(digits) - 4) + digits[-4:]

customers["Phone_Masked"] = (
    customers["Phone"]
    .apply(mask_phone)
)

display(
    customers[
        ["Phone", "Phone_Masked"]
    ].head()
)


# 10. Date Generalization

Exact dates of birth are usually unnecessary for aggregate CRM analytics.

We derive `Age_Group` instead of exposing the exact birth date.


In [ ]:
REFERENCE_DATE = pd.Timestamp("2026-08-25")

customers["Age"] = (
    (
        REFERENCE_DATE -
        customers["Birth_Date"]
    ).dt.days / 365.25
).astype(int)

age_bins = [0, 24, 34, 44, 54, 64, 200]

age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

customers["Age_Group"] = pd.cut(
    customers["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

display(
    customers[
        ["Birth_Date", "Age", "Age_Group"]
    ].head()
)


# 11. Build the Analytics-Safe Customer Layer

The analytics-safe layer removes direct identifiers that are unnecessary for analytical use.

### Removed
- Customer_ID
- First_Name
- Last_Name
- Email
- Phone
- Birth_Date
- exact Age

### Retained or Derived
- Customer_Token
- Age_Group
- State
- Signup_Date
- Marketing_Consent
- Customer_Segment
- Purchase metrics


In [ ]:
analytics_safe_customer = customers[
    [
        "Customer_Token",
        "Age_Group",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
].copy()

display(analytics_safe_customer.head())


# 12. Exposure Comparison

This section compares the raw and analytics-safe layers.


In [ ]:
raw_columns = set(customers.columns)

safe_columns = set(
    analytics_safe_customer.columns
)

removed_from_analytics = sorted(
    raw_columns - safe_columns
)

print("Fields excluded from the analytics-safe layer:")
for field in removed_from_analytics:
    print("-", field)


# 13. Privacy Control Catalog

Each privacy control receives an ID and rationale to make the framework auditable.


In [ ]:
privacy_control_catalog = pd.DataFrame([
    [
        "PRIV-001",
        "Data Minimization",
        "Remove customer names from analytics layer",
        "First_Name, Last_Name",
        "REMOVE",
        "Names are not required for aggregate analytics."
    ],
    [
        "PRIV-002",
        "Pseudonymization",
        "Replace Customer_ID with Customer_Token",
        "Customer_ID",
        "PSEUDONYMIZE",
        "Allows analytical linkage without exposing the source identifier."
    ],
    [
        "PRIV-003",
        "Masking",
        "Mask customer email",
        "Email",
        "MASK",
        "Reduces direct identifier exposure where limited display is needed."
    ],
    [
        "PRIV-004",
        "Masking",
        "Mask customer phone",
        "Phone",
        "MASK",
        "Reduces direct identifier exposure."
    ],
    [
        "PRIV-005",
        "Generalization",
        "Replace exact birth date with age group",
        "Birth_Date",
        "GENERALIZE",
        "Exact date of birth is unnecessary for aggregate analytics."
    ],
    [
        "PRIV-006",
        "Consent Governance",
        "Retain marketing consent as a controlled governance attribute",
        "Marketing_Consent",
        "KEEP / RESTRICT",
        "Consent status is required for governance and marketing eligibility analysis."
    ],
    [
        "PRIV-007",
        "Purpose Limitation",
        "Expose only analytics-required fields to the analytical layer",
        "Multiple Fields",
        "MINIMIZE",
        "Limits access to data necessary for the analytical use case."
    ]
], columns=[
    "Control_ID",
    "Control_Type",
    "Control_Description",
    "Field",
    "Treatment",
    "Rationale"
])

display(privacy_control_catalog)


# 14. Privacy Classification Coverage

A privacy program should also measure whether all fields have been classified.


In [ ]:
source_fields = set(
    [
        "Customer_ID",
        "First_Name",
        "Last_Name",
        "Email",
        "Phone",
        "Birth_Date",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
)

classified_fields = set(
    privacy_classification["Field_Name"]
)

unclassified_fields = sorted(
    source_fields - classified_fields
)

classification_coverage = (
    len(classified_fields.intersection(source_fields))
    /
    len(source_fields)
    * 100
)

print(
    f"Privacy classification coverage: "
    f"{classification_coverage:.2f}%"
)

print("Unclassified fields:", unclassified_fields)


# 15. Sensitive Field Inventory

In [ ]:
sensitive_inventory = (
    privacy_classification.groupby(
        ["Sensitivity", "Privacy_Class"]
    )
    .agg(
        Fields=("Field_Name", "count")
    )
    .reset_index()
)

display(sensitive_inventory)


# 16. Analytics Necessity Review

This view identifies fields that are sensitive but still required for analytics.


In [ ]:
analytics_necessity = privacy_classification[
    privacy_classification[
        "Required_for_Analytics"
    ] == True
][
    [
        "Field_Name",
        "Privacy_Class",
        "Sensitivity",
        "Recommended_Treatment"
    ]
]

display(analytics_necessity)


# 17. Privacy Risk Flags at Field Level

A simple field-level governance flag helps prioritize privacy engineering effort.


In [ ]:
privacy_risk_weights = {
    "Internal": 1,
    "Confidential": 2,
    "Restricted": 3
}

privacy_classification["Sensitivity_Weight"] = (
    privacy_classification[
        "Sensitivity"
    ].map(privacy_risk_weights)
)

privacy_classification["Direct_Identifier_Flag"] = (
    privacy_classification[
        "Privacy_Class"
    ].eq("Direct Identifier")
)

privacy_classification["Privacy_Priority_Score"] = (
    privacy_classification[
        "Sensitivity_Weight"
    ]
    +
    privacy_classification[
        "Direct_Identifier_Flag"
    ].astype(int) * 2
)

display(
    privacy_classification.sort_values(
        "Privacy_Priority_Score",
        ascending=False
    )
)


# 18. Link Privacy Controls to Access Governance

Privacy protection and access governance should reinforce each other.

Examples:

- pseudonymized customer data can be exposed more broadly than direct identifiers;
- restricted fields should require stronger role/action controls;
- exports involving direct identifiers should receive stricter treatment;
- analytics users should consume the safe layer instead of raw CRM records.


In [ ]:
privacy_access_mapping = pd.DataFrame([
    [
        "Restricted Direct Identifiers",
        "Admin / Authorized Operations Only",
        "Raw CRM Layer",
        "BLOCK or REVIEW for broad export"
    ],
    [
        "Masked Contact Data",
        "Limited Operational Roles",
        "Protected Operational Layer",
        "REVIEW for bulk access"
    ],
    [
        "Pseudonymized Customer Data",
        "Analytics / Governance Roles",
        "Analytics-Safe Layer",
        "ALLOW under approved purpose"
    ],
    [
        "Aggregate Customer Metrics",
        "Analytics / Management",
        "Analytics Layer",
        "ALLOW under normal governance"
    ]
], columns=[
    "Data_Category",
    "Recommended_Access",
    "Data_Layer",
    "Governance_Treatment"
])

display(privacy_access_mapping)


# 19. Proposed Privacy Architecture

```text
RAW CRM CUSTOMER DATA
        |
        v
PII / PRIVACY CLASSIFICATION
        |
        v
DATA MINIMIZATION
        |
  +-----+-------------------+
  |                         |
  v                         v
MASK / PSEUDONYMIZE      REMOVE
  |                         |
  +-----------+-------------+
              |
              v
      ANALYTICS-SAFE LAYER
              |
              v
      GOVERNED ANALYTICS
```

The original direct identifiers remain restricted to the operational CRM layer.


# 20. Privacy Monitoring KPIs

Future governance monitoring can include:

- Privacy Classification Coverage %
- Restricted Fields
- Direct Identifier Count
- Fields Removed from Analytics
- Fields Masked
- Fields Pseudonymized
- Analytics-Safe Fields
- Consent Coverage %
- Customers Without Marketing Consent
- Privacy Controls Implemented
- Privacy Control Exceptions


In [ ]:
privacy_kpis = pd.DataFrame({
    "Metric": [
        "Customer Records",
        "Classified Source Fields",
        "Direct Identifier Fields",
        "Restricted Fields",
        "Analytics-Safe Fields",
        "Marketing Consent Rate"
    ],
    "Value": [
        len(customers),
        len(classified_fields),
        privacy_classification[
            "Privacy_Class"
        ].eq("Direct Identifier").sum(),
        privacy_classification[
            "Sensitivity"
        ].eq("Restricted").sum(),
        analytics_safe_customer.shape[1],
        customers[
            "Marketing_Consent"
        ].mean() * 100
    ]
})

display(privacy_kpis.round(2))


# 21. Exportable Privacy Artifacts

The following DataFrames are intended to become persistent project artifacts:

- `customers`
- `privacy_classification`
- `minimization_matrix`
- `analytics_safe_customer`
- `privacy_control_catalog`
- `privacy_access_mapping`
- `privacy_kpis`

Suggested GitHub structure:

```text
governance/
├── privacy_classification.csv
├── privacy_control_catalog.csv
├── privacy_access_mapping.csv
└── data_minimization_matrix.csv

data/
├── raw/
│   └── synthetic_customers.csv
└── processed/
    └── analytics_safe_customers.csv
```


# 22. Findings to Document

## Privacy Classification
- Total customer fields:
- Direct identifiers:
- Indirect identifiers:
- Restricted fields:
- Classification coverage:

## Minimization
- Fields removed:
- Fields retained:
- Fields transformed:

## Privacy Engineering
- Pseudonymized fields:
- Masked fields:
- Generalized fields:

## Consent
- Marketing consent rate:
- Customers without consent:

## Governance Conclusions
1.
2.
3.


# 23. Limitations

1. Customer data is fully synthetic.
2. No real legal assessment is performed.
3. Hashing is used only to demonstrate pseudonymization concepts.
4. Secure key management and token vaults are not implemented.
5. No real retention policy is available.
6. No subject-rights workflow is implemented yet.
7. No automated PII scanner is used.
8. Privacy classifications are project-level governance classifications.


# 24. Next Step — Data Lineage & Traceability

## Stage 7 — Data Lineage & Traceability

Planned outputs:

- source-to-target mapping;
- transformation inventory;
- field-level lineage;
- relationship between source data and governance rules;
- relationship between privacy controls and analytical outputs;
- traceability from raw CRM to Power BI metrics;
- lineage documentation for GitHub.

This stage will make the project easier to audit and explain end to end.
